In [ ]:
# Lite LoRA config sizing (r=32, attention layers only)
r = 32
lora_q_proj = 3072 * r + 3072 * r
lora_k_proj = 3072 * r + 1024 * r
lora_v_proj = 3072 * r + 1024 * r
lora_o_proj = 3072 * r + 3072 * r
lora_layer = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj
params = lora_layer * 28  # 28 transformer layers in Llama-3.2-3B
size_mb = (params * 4) / 1_000_000

print(f"Trainable LoRA params: {params:,}")
print(f"Adapter size (fp32): {size_mb:.1f} MB")

Trainable LoRA params: 18,350,080
Adapter size (fp32): 73.4 MB


In [ ]:
# Full LoRA config sizing (r=256, attention + MLP layers)
r = 256

# attention layers (same shape as before, bigger r)
lora_q_proj = 3072 * r + 3072 * r
lora_k_proj = 3072 * r + 1024 * r
lora_v_proj = 3072 * r + 1024 * r
lora_o_proj = 3072 * r + 3072 * r

# MLP layers (new for the full config)
lora_gate_proj = 3072 * r + 8192 * r
lora_up_proj = 3072 * r + 8192 * r
lora_down_proj = 3072 * r + 8192 * r

lora_layer_full = (lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj +
                    lora_gate_proj + lora_up_proj + lora_down_proj)
params_full = lora_layer_full * 28
size_mb_full = (params_full * 4) / 1_000_000

print(f"Trainable LoRA params (full config): {params_full:,}")
print(f"Adapter size (fp32): {size_mb_full:.1f} MB")
print(f"\nRatio full/lite: {params_full / params:.1f}x")

Trainable LoRA params (full config): 389,021,696
Adapter size (fp32): 1556.1 MB

Ratio full/lite: 21.2x


In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login

BASE_MODEL = "meta-llama/Llama-3.2-3B"

# Colab doesn't have your local .env — you need to auth here separately.
# Use Colab's Secrets manager (key icon in left sidebar) to store HF_TOKEN,
# then this will pick it up. Or paste it directly (less secure, fine for now).
from google.colab import userdata
login(token=userdata.get('HF_TOKEN'))

In [ ]:
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e9:,.1f} GB")

config.json:   0%|          | 0.00/844 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/20.9k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

Memory footprint: 6.4 GB


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

BASE_MODEL = "meta-llama/Llama-3.2-3B"

quant_config = BitsAndBytesConfig(load_in_8bit=True)
base_model_8bit = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", quantization_config=quant_config
)
print(f"Memory footprint (8-bit): {base_model_8bit.get_memory_footprint() / 1e9:,.1f} GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Memory footprint (8-bit): 3.6 GB


In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

BASE_MODEL = "meta-llama/Llama-3.2-3B"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)
base_model_4bit = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, device_map="auto", quantization_config=quant_config
)
print(f"Memory footprint (4-bit): {base_model_4bit.get_memory_footprint() / 1e9:,.1f} GB")

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


Memory footprint (4-bit): 2.2 GB
